# 00 — Data Prep And Split Audit

Load CICIoT2023 + TON_IoT, do the **mechanism-critical** cleaning (dedup + identity-field removal), run the **group-count audit**, and lock the grouped splits. RESOLVES two freeze-checklist items: grouping variable + 34-vs-8 granularity.

In [2]:
import pandas as pd, glob, os

# point these at wherever your CSVs actually sit in Drive
SEARCH = {
    "ciciot2023": "/content/drive/MyDrive/**/CICIoT2023/**/*.csv",
    "ton_iot":    "/content/drive/MyDrive/**/TON_IoT/**/*.csv",
}
for name, pat in SEARCH.items():
    files = glob.glob(pat, recursive=True)
    print(f"\n===== {name}: {len(files)} csv files =====")
    for f in files[:3]:
        print(" ", f.replace("/content/drive/MyDrive/", ".../"))
    if files:
        df = pd.read_csv(files[0], nrows=500)
        print("  shape (first file, 500 rows):", df.shape)
        print("  columns:", list(df.columns))
        # find the label-ish column and show its values
        lab = [c for c in df.columns if c.lower() in
               ("label","attack","category","type","class","traffic_type")]
        print("  candidate label column(s):", lab)
        if lab:
            print("  label values:", df[lab[0]].astype(str).unique()[:20])


===== ciciot2023: 0 csv files =====

===== ton_iot: 0 csv files =====


In [1]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


Mounted at /content/drive


## Load + clean (identity removal, dedup FIRST)

Leaked identifiers create spurious separability that compression 'loses' — clean before anything else.

In [4]:
import os, shutil, glob

# find the uploaded kaggle.json wherever it landed
src = "kaggle.json" if os.path.exists("kaggle.json") else glob.glob("/content/**/kaggle.json", recursive=True)[0]
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(src, "/root/.kaggle/kaggle.json")     # copy, not move — works across devices
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip install -q kaggle

# verify auth works, then search for live slugs
print("\n=== auth check ===")
!kaggle datasets list -s "CICIoT2023" --csv | head -1

print("\n=== CICIoT2023 candidates ===")
!kaggle datasets list -s "CICIoT2023" --csv | head -8
print("\n=== TON_IoT network candidates ===")
!kaggle datasets list -s "TON_IoT network" --csv | head -8


=== auth check ===
ref,title,size,lastUpdated,downloadCount,voteCount,usabilityRating

=== CICIoT2023 candidates ===
ref,title,size,lastUpdated,downloadCount,voteCount,usabilityRating
himadri07/ciciot2023,CICIoT2023,500559747,2025-04-21 15:38:38.783000,5512,30,0.7647059
subhajournal/iotintrusion,IoT Intrusion Detection,47315907,2023-07-16 06:30:21.843000,5068,37,0.9411765
mastole/ddos-ciciot2023,DDoS-CICIoT2023,2028434345,2024-06-04 02:28:44.143000,632,1,0.4117647
daiqing2009/ciciot2023-disprop-sampling,CICIOT2023-disprop-sampling,1031063411,2024-12-07 00:25:57.300000,223,0,0.7647059
zaywreck/ciciot2023,ciciot2023,1789835747,2026-04-06 20:17:27.537000,23,1,0.11764706
ioannispan/ciciot2023-1m,CICIoT2023-1M,83573555,2024-12-07 14:34:52.263000,100,0,0.29411766
nikitamanaenkov/large-scale-attacks-in-iot-environment,Large-Scale Attacks in IoT Environment,1474647877,2025-05-07 19:30:21.110000,243,5,1

=== TON_IoT network candidates ===
ref,title,size,lastUpdated,downloadCount,voteCount,usab

In [5]:
DATA = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/data/raw"

# CICIoT2023 — full merged flow CSVs, true class imbalance intact
!kaggle datasets download -d himadri07/ciciot2023 -p "{DATA}/CICIoT2023" --unzip

# TON_IoT — curated network subset (Train_Test_Network.csv)
!kaggle datasets download -d arnobbhowmik/ton-iot-network-dataset -p "{DATA}/TON_IoT" --unzip

# confirm what landed
import glob
for name in ["CICIoT2023", "TON_IoT"]:
    fs = glob.glob(f"{DATA}/{name}/**/*.csv", recursive=True)
    print(f"\n{name}: {len(fs)} CSV(s)")
    for f in fs[:8]: print("  ", f.split('/')[-1])

Dataset URL: https://www.kaggle.com/datasets/himadri07/ciciot2023
License(s): MIT
100% 477M/477M [00:06<00:00, 82.5MB/s]

Dataset URL: https://www.kaggle.com/datasets/arnobbhowmik/ton-iot-network-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.71M/1.71M [00:00<00:00, 11.0MB/s]


CICIoT2023: 3 CSV(s)
   test.csv
   train.csv
   validation.csv

TON_IoT: 1 CSV(s)
   train_test_network.csv


In [6]:
import pandas as pd, glob
DATA = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/data/raw"

# ---- CICIoT2023 ----
ci_files = sorted(glob.glob(f"{DATA}/CICIoT2023/**/*.csv", recursive=True))
print("=== CICIoT2023 ===")
for f in ci_files:
    df = pd.read_csv(f, nrows=5)
    print(f"\n{f.split('/')[-1]}  ncols={df.shape[1]}")
print("\ncolumns:", list(df.columns))
lab = [c for c in df.columns if c.lower() in
       ("label","attack","category","type","class","traffic_type")]
print("label column(s):", lab)
# how many distinct classes, and at what granularity? (read full train label only)
if lab:
    tr = [f for f in ci_files if "train" in f.lower()][0]
    y = pd.read_csv(tr, usecols=[lab[0]])[lab[0]].astype(str)
    print(f"\nN classes in train: {y.nunique()}")
    print("class counts:\n", y.value_counts())

# ---- TON_IoT ----
print("\n\n=== TON_IoT ===")
t = pd.read_csv(f"{DATA}/TON_IoT/train_test_network.csv", nrows=5)
print("ncols:", t.shape[1])
print("columns:", list(t.columns))
tlab = [c for c in t.columns if c.lower() in ("label","type","attack_cat","attack")]
print("label column(s):", tlab)
for c in tlab:
    vals = pd.read_csv(f"{DATA}/TON_IoT/train_test_network.csv", usecols=[c])[c].astype(str)
    print(f"  {c}: {vals.nunique()} values ->", vals.value_counts().to_dict())

=== CICIoT2023 ===

test.csv  ncols=47

train.csv  ncols=47

validation.csv  ncols=47

columns: ['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate', 'Srate', 'Drate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'urg_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Magnitue', 'Radius', 'Covariance', 'Variance', 'Weight', 'label']
label column(s): ['label']

N classes in train: 34
class counts:
 label
DDoS-ICMP_Flood            848088
DDoS-UDP_Flood             637558
DDoS-TCP_Flood             528499
DDoS-PSHACK_Flood          481254
DDoS-SYN_Flood             478653
DDoS-RSTFINFlood           475441
DDoS-SynonymousIP_Flood    422083
DoS-UDP_Flood              390422
DoS-TCP_Flood              314

In [7]:
import subprocess, re

def files_of(slug):
    out = subprocess.run(["kaggle","datasets","files","-v",slug],
                         capture_output=True, text=True).stdout
    rows = [r.split(",")[0] for r in out.splitlines()[1:] if r.strip()]
    return rows

# gather candidates from several search phrasings
queries = ["CICIoT2023", "CIC IoT 2023", "CICIoT 2023 dataset", "IoT 2023 intrusion CICFlowMeter"]
slugs = set()
for q in queries:
    out = subprocess.run(["kaggle","datasets","list","-s",q,"--csv","--page-size","30"],
                         capture_output=True, text=True).stdout
    for line in out.splitlines()[1:]:
        if line.strip(): slugs.add(line.split(",")[0])

print(f"Inspecting {len(slugs)} candidate mirrors...\n")
ATTACK = re.compile(r"(ddos|dos|mirai|recon|spoof|benign|brute|inject|xss|backdoor|scan|flood|http|arp|sql)", re.I)

scored = []
for s in sorted(slugs):
    fs = files_of(s)
    csvs = [f for f in fs if f.lower().endswith(".csv")]
    named = [f for f in csvs if ATTACK.search(f)]
    merged_like = any(re.search(r"(^|/)(train|test|val|merged|full|all|combined|1m)\b", f, re.I) for f in csvs)
    # multi-file per-attack release = many CSVs, attack-named, not a train/test triple
    is_provenance = len(csvs) >= 20 and len(named) >= 10
    score = (is_provenance, len(csvs), len(named))
    scored.append((score, s, len(csvs), len(named), merged_like, csvs[:4]))

scored.sort(reverse=True)
for (prov,*_), s, ncsv, nnamed, merged, sample in scored:
    flag = "PROVENANCE-PRESERVING (multi-file)" if prov else ("merged/split — no grouping key" if merged or ncsv<=5 else "unclear")
    print(f"[{flag}]\n  {s}  | {ncsv} CSVs, {nnamed} attack-named")
    print(f"  sample files: {sample}\n")

Inspecting 0 candidate mirrors...



In [8]:
import subprocess, re

def files_of(slug):
    r = subprocess.run(["kaggle","datasets","files",slug],
                       capture_output=True, text=True)
    return r.stdout, r.stderr

ATTACK = re.compile(r"(ddos|dos|mirai|recon|spoof|benign|brute|inject|xss|backdoor|scan|flood|arp|sql|web|vulnerab)", re.I)

candidates = [
    "nikitamanaenkov/large-scale-attacks-in-iot-environment",
    "zaywreck/ciciot2023",
    "mastole/ddos-ciciot2023",
    "madhavmalhotra/unb-cic-iot-2023",
    "akashdogra/cic-iot-2023",
    "subhajournal/iotintrusion",
]

for s in candidates:
    out, err = files_of(s)
    if err.strip() and "not found" in err.lower():
        print(f"--- {s} : NOT FOUND\n"); continue
    lines = [l for l in out.splitlines()[1:] if l.strip()]
    csvs  = [l.split(",")[0] for l in lines if ".csv" in l.lower()]
    named = [f for f in csvs if ATTACK.search(f)]
    verdict = "MULTI-FILE (provenance OK)" if len(csvs) >= 15 and len(named) >= 8 else \
              ("merged/split — no grouping key" if csvs else "no CSVs / odd structure")
    print(f"--- {s} : {len(csvs)} CSVs, {len(named)} attack-named  ->  {verdict}")
    for f in csvs[:6]: print("      ", f)
    print()

--- nikitamanaenkov/large-scale-attacks-in-iot-environment : 20 CSVs, 19 attack-named  ->  MULTI-FILE (provenance OK)
       data/Backdoor_Malware/Backdoor_Malware.pcap.csv                  664577  2025-05-07 19:30:28.770000  
       data/Benign_Final/BenignTraffic.pcap.csv                       75100977  2025-05-07 19:30:34.829000  
       data/Benign_Final/BenignTraffic1.pcap.csv                      61260813  2025-05-07 19:30:34.659000  
       data/Benign_Final/BenignTraffic2.pcap.csv                      64200745  2025-05-07 19:30:34.160000  
       data/Benign_Final/BenignTraffic3.pcap.csv                      26851364  2025-05-07 19:30:33.761000  
       data/BrowserHijacking/BrowserHijacking.pcap.csv                 1207500  2025-05-07 19:30:33.292000  

--- zaywreck/ciciot2023 : 20 CSVs, 0 attack-named  ->  merged/split — no grouping key
       MERGED_CSV/Merged01.csv  147115614  2026-04-06 20:18:26.620000  
       MERGED_CSV/Merged02.csv  154639360  2026-04-06 20:18:34.277000

In [9]:
DATA = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/data/raw"

# provenance-preserving multi-file CICIoT2023 (per-attack capture folders)
!kaggle datasets download -d nikitamanaenkov/large-scale-attacks-in-iot-environment -p "{DATA}/CICIoT2023_multifile" --unzip

import glob, os
fs = glob.glob(f"{DATA}/CICIoT2023_multifile/**/*.csv", recursive=True)
folders = sorted(set(os.path.basename(os.path.dirname(f)) for f in fs))
print(f"{len(fs)} CSVs across {len(folders)} capture folders\n")
print("capture folders (= grouping units):")
for d in folders: print("  ", d)

Dataset URL: https://www.kaggle.com/datasets/nikitamanaenkov/large-scale-attacks-in-iot-environment
License(s): Attribution 4.0 International (CC BY 4.0)
100% 1.37G/1.37G [00:24<00:00, 60.6MB/s]

309 CSVs across 34 capture folders

capture folders (= grouping units):
   Backdoor_Malware
   Benign_Final
   BrowserHijacking
   CommandInjection
   DDoS-ACK_Fragmentation
   DDoS-HTTP_Flood
   DDoS-ICMP_Flood
   DDoS-ICMP_Fragmentation
   DDoS-PSHACK_FLOOD
   DDoS-RSTFINFLOOD
   DDoS-SYN_Flood
   DDoS-SlowLoris
   DDoS-SynonymousIP_Flood
   DDoS-TCP_Flood
   DDoS-UDP_Flood
   DDoS-UDP_Fragmentation
   DNS_Spoofing
   DictionaryBruteForce
   DoS-HTTP_Flood
   DoS-SYN_Flood
   DoS-TCP_Flood
   DoS-UDP_Flood
   MITM-ArpSpoofing
   Mirai-greeth_flood
   Mirai-greip_flood
   Mirai-udpplain
   Recon-HostDiscovery
   Recon-OSScan
   Recon-PingSweep
   Recon-PortScan
   SqlInjection
   Uploading_Attack
   VulnerabilityScan
   XSS


In [10]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
import os; os.makedirs(f"{REPO}/src", exist_ok=True)
SRC = r'''
"""
src/data.py — loading, leakage-critical cleaning, and the splits.

Coded against the provenance-preserving multi-file CICIoT2023 release
(nikitamanaenkov/large-scale-attacks-in-iot-environment): one folder per attack
type under data/raw/CICIoT2023_multifile/, each holding one or more
<...>.pcap.csv part-files with 46 CICFlowMeter features + a 'label' column.
Each row is stamped with its capture-folder provenance as 'group'.

SPLIT DECISION (frozen): each attack type was captured separately, so capture
file confounds with the class label. Therefore:
  * PRIMARY  = temporal_within_capture: split each capture's rows by capture
    order (row order proxies capture time). Breaks near-duplicate-burst leakage
    while keeping EVERY rare class evaluable on both sides.
  * ROBUSTNESS = strict_capture_grouped: whole captures to one side; only runs
    on classes with >= 2 capture part-files (essentially Benign) — confirms the
    effect is not an artifact of the temporal-order assumption.
  * REFERENCE = random stratified split, kept ONLY to measure the leakage gap.

Dedup + identity-field removal are MECHANISM-CRITICAL (leaked identifiers create
spurious separability that compression 'loses', masquerading as capacity loss).
"""
from __future__ import annotations
from pathlib import Path
import glob, os, re
import numpy as np
import pandas as pd

from .config import CFG, PATHS

# 46 CICFlowMeter features (everything except the label). Confirmed schema.
LABEL_COL = "label"
GROUP_COL = "group"

# CICIoT2023 multi-file rows carry no in-row identity fields (flow IDs/IPs/timestamps
# were not exported), so identity removal is a no-op there but kept for symmetry.
IDENTITY_FIELDS = {
    "ciciot2023": [],
    "ton_iot": ["src_ip", "dst_ip", "src_port", "dst_port",
                "dns_query", "ssl_subject", "ssl_issuer",
                "http_uri", "http_user_agent",
                "weird_name", "weird_addl", "weird_notice"],
}

# folder spellings vary in case (DDoS-PSHACK_FLOOD vs ..._Flood) -> canonicalise
_CANON = {
    "ddos-pshack_flood": "DDoS-PSHACK_Flood",
    "ddos-rstfinflood": "DDoS-RSTFINFlood",
    "ddos-syn_flood": "DDoS-SYN_Flood",
    "benign_final": "BenignTraffic",
}
def _canon_label(folder: str) -> str:
    return _CANON.get(folder.lower(), folder)


def load_raw(dataset: str, subsample: bool = True, seed: int | None = None) -> pd.DataFrame:
    """Load a dataset into one DataFrame with 'label' and (for ciciot2023) 'group'.

    subsample=True (default, sensible on Colab Pro): cap each MAJORITY class while
    keeping EVERY rare class whole, so the long-tail phenomenon is intact but the
    working set is fast to iterate on. Caps are read from CFG. Run with
    subsample=False once at the end for the full-scale confirmation pass.
    """
    seed = CFG["anchor_seed"] if seed is None else seed
    if dataset == "ciciot2023":
        return _load_ciciot_multifile(subsample=subsample, seed=seed)
    if dataset == "ton_iot":
        return _load_ton(subsample=subsample, seed=seed)
    raise ValueError(dataset)


def _load_ciciot_multifile(subsample: bool, seed: int) -> pd.DataFrame:
    root = PATHS.data("raw", "CICIoT2023_multifile")
    folders = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    if not folders:
        raise FileNotFoundError(f"No capture folders under {root} — check the download path.")

    cap = CFG.get("subsample", {}).get("majority_cap_per_class", 200_000)
    rng = np.random.default_rng(seed)
    frames = []
    for folder in folders:
        files = sorted(glob.glob(os.path.join(root, folder, "*.csv")))
        parts = []
        for fi, f in enumerate(files):
            df = pd.read_csv(f)
            # capture-order proxy: preserve part-file index then within-file row order
            df["_part"] = fi
            df["_row"] = np.arange(len(df))
            parts.append(df)
        cls = pd.concat(parts, ignore_index=True)
        cls[LABEL_COL] = _canon_label(folder)
        cls[GROUP_COL] = folder                      # provenance = capture folder
        # subsample MAJORITY classes only; keep rare classes whole
        if subsample and len(cls) > cap:
            cls = cls.iloc[rng.choice(len(cls), cap, replace=False)].reset_index(drop=True)
        frames.append(cls)
    df = pd.concat(frames, ignore_index=True)
    return df


def _load_ton(subsample: bool, seed: int) -> pd.DataFrame:
    f = PATHS.data("raw", "TON_IoT", "train_test_network.csv")
    df = pd.read_csv(f)
    # Use the MULTICLASS 'type' as the canonical label. Drop the binary 'label'
    # FIRST, then rename — otherwise renaming 'type'->'label' collides with the
    # existing binary 'label' and creates two identically named columns.
    if "label" in df.columns:
        df = df.drop(columns=["label"])
    df = df.rename(columns={"type": LABEL_COL})
    df[GROUP_COL] = "ton_single_capture"            # TON network ships as one curated file
    # order helpers for uniformity with the multifile loader (TON = single capture, so
    # one part; the 'temporal' split degenerates to a within-class row-order split here.
    # NOTE: the curated TON file may not preserve true capture order, so for TON a
    # random/stratified split is arguably more appropriate — revisit at cross-dataset stage.)
    df["_part"] = 0
    df["_row"] = np.arange(len(df))
    return df


def clean(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Identity-field removal + dedup. MECHANISM-CRITICAL. Done on FEATURES only;
    label/group/order-helper columns are preserved."""
    keep_meta = [c for c in (LABEL_COL, GROUP_COL, "_part", "_row") if c in df.columns]
    drop = [c for c in IDENTITY_FIELDS.get(dataset, []) if c in df.columns]
    df = df.drop(columns=drop)
    # numeric coercion + drop rows with inf/NaN introduced by CICFlowMeter
    feat = [c for c in df.columns if c not in keep_meta]
    df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feat).reset_index(drop=True)
    # dedup on FEATURES + label (near-duplicate flow rows). Keep order helpers out of the key.
    dedup_key = [c for c in df.columns if c not in ("_part", "_row")]
    df = df.drop_duplicates(subset=dedup_key).reset_index(drop=True)
    return df


def group_count_audit(df: pd.DataFrame, label_col: str = LABEL_COL) -> pd.DataFrame:
    """Per-class instance count and capture-group count, with the survival verdict.
    For THIS dataset 'groups' = capture part-files within the class (≈ how many
    chunks the class was captured in). Drives the 34-vs-survivors reality check and
    flags which classes can take the STRICT capture-grouped robustness split."""
    surv = CFG["split"]["fine_type_survival"]
    g = df.groupby(label_col).agg(
        n_instances=("_row", "size"),
        n_capture_parts=("_part", lambda s: s.nunique()),
    )
    g["survives_min_instances"] = g["n_instances"] >= surv["min_train_instances"]
    g["can_strict_group"] = g["n_capture_parts"] >= surv["min_groups"]   # >=2 parts
    return g.sort_values("n_instances")


def temporal_within_capture_split(df, seed: int, label_col: str = LABEL_COL):
    """PRIMARY split. Within each (class), order by capture (_part,_row) and cut
    chronologically into train/val/test by config fractions. Near-duplicate bursts
    stay on one side; every class appears on all sides. Returns index arrays."""
    tr, va, te = (CFG["split"][k] for k in ("train_frac", "val_frac", "test_frac"))
    train_idx, val_idx, test_idx = [], [], []
    for _, sub in df.groupby(label_col):
        order = sub.sort_values(["_part", "_row"]).index.to_numpy()
        n = len(order); a, b = int(n * tr), int(n * (tr + va))
        train_idx += order[:a].tolist(); val_idx += order[a:b].tolist(); test_idx += order[b:].tolist()
    return {"train": np.array(train_idx), "val": np.array(val_idx), "test": np.array(test_idx)}


def strict_capture_grouped_split(df, seed: int, label_col: str = LABEL_COL):
    """ROBUSTNESS split. Whole capture-parts to one side; only meaningful for classes
    with >= 2 parts (others land entirely in train and are reported as excluded from
    this check). Returns index arrays + the list of classes actually evaluated."""
    surv = CFG["split"]["fine_type_survival"]
    rng = np.random.default_rng(seed)
    train_idx, test_idx, evaluated = [], [], []
    for cls, sub in df.groupby(label_col):
        parts = sorted(sub["_part"].unique())
        if len(parts) >= surv["min_groups"]:
            rng.shuffle(parts)
            k = max(1, int(len(parts) * CFG["split"]["test_frac"]))
            test_parts = set(parts[:k]); evaluated.append(cls)
            test_idx += sub[sub["_part"].isin(test_parts)].index.tolist()
            train_idx += sub[~sub["_part"].isin(test_parts)].index.tolist()
        else:
            train_idx += sub.index.tolist()   # single-capture class: train-only here
    return {"train": np.array(train_idx), "test": np.array(test_idx), "evaluated_classes": evaluated}


def random_reference_split(df, seed: int, label_col: str = LABEL_COL):
    """REFERENCE ONLY — stratified random split to MEASURE the leakage gap vs the
    temporal split. Never the headline regime."""
    from sklearn.model_selection import train_test_split
    tr, va, te = (CFG["split"][k] for k in ("train_frac", "val_frac", "test_frac"))
    idx = np.arange(len(df)); y = df[label_col].to_numpy()
    tr_i, tmp_i = train_test_split(idx, train_size=tr, stratify=y, random_state=seed)
    rel = te / (va + te)
    va_i, te_i = train_test_split(tmp_i, test_size=rel, stratify=y[tmp_i], random_state=seed)
    return {"train": tr_i, "val": va_i, "test": te_i}
'''
open(f"{REPO}/src/data.py","w").write(SRC)
print("wrote src/data.py:", len(SRC), "chars  (expect ~7700)")
assert "label" in open(f"{REPO}/src/data.py").read()
print("OK — fixed data.py written to Drive")

wrote src/data.py: 9726 chars  (expect ~7700)
OK — fixed data.py written to Drive


In [11]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
import os; os.makedirs(f"{REPO}/config", exist_ok=True)
CFG_YAML = r'''
# =====================================================================
# iot-trust-compression — single source of truth
# Every notebook and module reads paths/seeds/splits FROM HERE.
# NO path is ever hardcoded inline (the X-IDS multi-seed lesson:
# inline Path(REPO)/... construction is what made the pipeline
# un-redirectable and destroyed the seed-42 baseline).
# This file mirrors anchor_preregistration.json. Keep them in sync.
# =====================================================================

project: iot-trust-compression
author: Md Anas Biswas

# --- Roots -----------------------------------------------------------
# In Colab the repo is cloned INSIDE Drive, so one tree serves both.
# Override drive_root at runtime if a notebook runs elsewhere.
paths:
  drive_root: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
  data: data                 # gitignored — lives in Drive only
  models: models             # gitignored — binaries in Drive only
  results: results           # tracked (tables/figures/arrays)
  tables: results/tables
  figures: results/figures
  arrays: results/arrays
  logs: logs

# --- Reproducibility -------------------------------------------------
seeds: [0, 1, 2, 3, 4]
anchor_seed: 0               # the pre-registered reference seed

# --- Datasets --------------------------------------------------------
# n_fine / n_categories per the prereg. diagnostic_unit drives the
# correlation granularity (DECISION: 34, see label_granularity).
datasets:
  ciciot2023:
    role: core
    n_fine: 34
    n_categories: 8
    diagnostic_unit: fine          # correlate the diagnostic over fine types
    drop_identity_fields: true
    deduplicate: true
  ton_iot:
    role: core
    drop_identity_fields: true     # addresses/ports/timestamps OUT of features
    deduplicate: true
  bot_iot:
    role: optional                 # held-out generalisation + robustness
    deduplicate: true

# --- Splits (INTEGRITY requirement, not optional) --------------------
split:
  primary: temporal_within_capture   # each attack captured separately -> capture confounds with class;
                                      # split each class chronologically (row order proxies capture time).
                                      # Breaks near-duplicate-burst leakage; keeps every rare class evaluable.
  robustness: strict_capture_grouped # whole captures to one side; only runs on classes with >=2 capture parts
  applies_to: [measure, crux, predict]
  train_frac: 0.70
  val_frac: 0.15
  test_frac: 0.15
  grouping_variable: capture_folder  # RESOLVED: provenance = per-attack capture folder/part (nikitamanaenkov multifile)
  capture_order_assumption: "row order within each capture CSV proxies capture time (CICFlowMeter writes in capture order); stated explicitly; strict_capture_grouped robustness check partly backs it"
  fine_type_survival:
    min_train_instances: 50      # learnable-count floor
    min_groups: 2                # >=2 capture parts to be eligible for the STRICT robustness split
  keep_random_reference: true    # random split retained to MEASURE the leakage gap
  temporal_split: robustness     # Stage 5 only; timestamps used to SPLIT, not as features

# --- Subsampling (Colab Pro: fast iteration set; full set for final confirmation) ---
subsample:
  majority_cap_per_class: 200000   # cap MAJORITY classes; rare classes kept WHOLE
  rationale: "rare-class collapse is driven by RELATIVE rarity, not absolute count; cap the head, keep the tail intact. Run load_raw(subsample=False) once at the end for a full-scale confirmation pass."

# --- Label granularity (FREEZE-CHECKLIST decision) -------------------
label_granularity:
  chosen: 34_way                 # CONFIRMED from data: all 34 fine classes present at true frequencies
  rule: >
    Prefer 34_way iff enough fine types survive the grouped split with
    >= min_train_instances and >= min_groups. Else keep per-type recall
    well-defined and correlate over surviving fine types; category-level
    only where group counts force it. Consistent across ALL units.

# --- Architectures (size- and macroF1-matched, edge-scale) -----------
match:
  on: macro_f1                   # NOT accuracy (accuracy hides collapse)
  tolerance: 0.02                # under the PRIMARY grouped split
architectures:
  mlp:
    role: dense_anchor
    params: null                 # FREEZE-CHECKLIST: from built anchor
    baseline_macro_f1: null
  cnn1d:
    role: local_conv_BUILD_FIRST # carries Stage 1 gate + crux experiment
    params: null
    baseline_macro_f1: null
  ft_transformer:
    role: attention_third_arm
    params: null
    baseline_macro_f1: null
    fallback: gru                # if cannot reach MLP macro-F1 +/- tol on BOTH datasets
    fallback_tuning_budget: 30   # trials before switching

# --- Compression matrix (PyTorch-native) -----------------------------
compression:
  matrix: [M0, prune50, prune80, distillation, int8, float16]
  prune_method: l1_unstructured     # torch.nn.utils.prune
  float16_method: half              # .half()
  int8_method: torchao_fx_ptq       # torchao / FX-graph PTQ
  int8_transformer_fallback: true   # int8 on MLP/CNN only if transformer PTQ won't yield attribution-accessible model

# --- Metrics ---------------------------------------------------------
metrics:
  bootstrap_B: 1000
  ece_bins: [10, 15, 20]            # adaptive equal-mass
  attribution_method: kernelshap    # perturbation-based across WHOLE matrix (DeepSHAP lies on int8)
  aopc_top_k: 5
  # explanation trust is decomposed: STABILITY (Spearman vs anchor) is NOT
  # faithfulness; FAITHFULNESS is adjudicated by deletion/AOPC ALONE.
  # drift is stratified by prediction-change. See src/metrics.py.

# --- Crux experiment -------------------------------------------------
crux:
  probe: linear_one_vs_rest
  auc_retention_threshold: null     # FREEZE before running; report on a continuum regardless
  interpretation: graded_class_dependent   # NOT a clean binary

# --- Diagnostic (headline; mechanism-grounded) -----------------------
diagnostic:
  target_primary: delta_recall      # continuous
  target_secondary: binary_collapse
  features:
    - sample_count
    - margin
    - separability
    - attribution_concentration     # FLAGGED redundancy-confounded; include only if it beats redundancy-free set
    - effective_rank
    - neural_collapse_geometry      # v2.1 theory-derived (ETF deviation / minority-collapse angle)
  baselines_to_beat:
    - frequency_only
    - tran_fioretto_margin_gradnorm_only
  generalisation: [across_architectures, across_compression_families, across_datasets]
  # across_architectures EARNS the causal claim; failure -> withdraw to correlational.

# --- Mitigation (closing payoff; droppable) --------------------------
mitigate:
  strategy: predictor_guided_selective_protection
  conformal_certificate: stretch    # class-conditional CRC; fallback to empirical risk control
  conformal_alpha: null             # set with feasibility gate (~1/alpha calib instances for rarest)
'''
open(f"{REPO}/config/config.yaml","w").write(CFG_YAML)
import yaml; c=yaml.safe_load(open(f"{REPO}/config/config.yaml"))
print("wrote config.yaml:", len(CFG_YAML), "chars")
print("split.primary =", c["split"]["primary"])
print("label_granularity.chosen =", c["label_granularity"]["chosen"])
print("subsample cap =", c["subsample"]["majority_cap_per_class"])
print("OK — config.yaml written to Drive")

wrote config.yaml: 7075 chars
split.primary = temporal_within_capture
label_granularity.chosen = 34_way
subsample cap = 200000
OK — config.yaml written to Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO)
sys.path.insert(0, REPO)
print("cwd:", os.getcwd())

# confirm the updated files are the ones loading
from src.config import CFG, PATHS
print("split.primary =", CFG['split']['primary'])          # -> temporal_within_capture
print("label_granularity =", CFG['label_granularity']['chosen'])  # -> 34_way
print("subsample cap =", CFG['subsample']['majority_cap_per_class'])  # -> 200000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
split.primary = temporal_within_capture
label_granularity = 34_way
subsample cap = 200000


In [4]:
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
print(data.group_count_audit(df).to_string())

ValueError: No objects to concatenate

In [5]:
import os, glob
root = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/data/raw/CICIoT2023_multifile"

print("exists:", os.path.isdir(root))
print("\n-- level 1 (what's directly inside CICIoT2023_multifile) --")
print(os.listdir(root))

# where do the CSVs actually live? show a few full paths and their depth
csvs = glob.glob(f"{root}/**/*.csv", recursive=True)
print(f"\ntotal CSVs found recursively: {len(csvs)}")
for f in csvs[:5]:
    rel = f.replace(root, "")
    print("   ", rel, "  (depth", rel.count("/"), ")")

exists: True

-- level 1 (what's directly inside CICIoT2023_multifile) --
['data']

total CSVs found recursively: 309
    /data/Backdoor_Malware/Backdoor_Malware.pcap.csv   (depth 3 )
    /data/Benign_Final/BenignTraffic.pcap.csv   (depth 3 )
    /data/Benign_Final/BenignTraffic1.pcap.csv   (depth 3 )
    /data/Benign_Final/BenignTraffic2.pcap.csv   (depth 3 )
    /data/Benign_Final/BenignTraffic3.pcap.csv   (depth 3 )


In [6]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/data.py — loading, leakage-critical cleaning, and the splits.

Coded against the provenance-preserving multi-file CICIoT2023 release
(nikitamanaenkov/large-scale-attacks-in-iot-environment): one folder per attack
type under data/raw/CICIoT2023_multifile/, each holding one or more
<...>.pcap.csv part-files with 46 CICFlowMeter features + a 'label' column.
Each row is stamped with its capture-folder provenance as 'group'.

SPLIT DECISION (frozen): each attack type was captured separately, so capture
file confounds with the class label. Therefore:
  * PRIMARY  = temporal_within_capture: split each capture's rows by capture
    order (row order proxies capture time). Breaks near-duplicate-burst leakage
    while keeping EVERY rare class evaluable on both sides.
  * ROBUSTNESS = strict_capture_grouped: whole captures to one side; only runs
    on classes with >= 2 capture part-files (essentially Benign) — confirms the
    effect is not an artifact of the temporal-order assumption.
  * REFERENCE = random stratified split, kept ONLY to measure the leakage gap.

Dedup + identity-field removal are MECHANISM-CRITICAL (leaked identifiers create
spurious separability that compression 'loses', masquerading as capacity loss).
"""
from __future__ import annotations
from pathlib import Path
import glob, os, re
import numpy as np
import pandas as pd

from .config import CFG, PATHS

# 46 CICFlowMeter features (everything except the label). Confirmed schema.
LABEL_COL = "label"
GROUP_COL = "group"

# CICIoT2023 multi-file rows carry no in-row identity fields (flow IDs/IPs/timestamps
# were not exported), so identity removal is a no-op there but kept for symmetry.
IDENTITY_FIELDS = {
    "ciciot2023": [],
    "ton_iot": ["src_ip", "dst_ip", "src_port", "dst_port",
                "dns_query", "ssl_subject", "ssl_issuer",
                "http_uri", "http_user_agent",
                "weird_name", "weird_addl", "weird_notice"],
}

# folder spellings vary in case (DDoS-PSHACK_FLOOD vs ..._Flood) -> canonicalise
_CANON = {
    "ddos-pshack_flood": "DDoS-PSHACK_Flood",
    "ddos-rstfinflood": "DDoS-RSTFINFlood",
    "ddos-syn_flood": "DDoS-SYN_Flood",
    "benign_final": "BenignTraffic",
}
def _canon_label(folder: str) -> str:
    return _CANON.get(folder.lower(), folder)


def load_raw(dataset: str, subsample: bool = True, seed: int | None = None) -> pd.DataFrame:
    """Load a dataset into one DataFrame with 'label' and (for ciciot2023) 'group'.

    subsample=True (default, sensible on Colab Pro): cap each MAJORITY class while
    keeping EVERY rare class whole, so the long-tail phenomenon is intact but the
    working set is fast to iterate on. Caps are read from CFG. Run with
    subsample=False once at the end for the full-scale confirmation pass.
    """
    seed = CFG["anchor_seed"] if seed is None else seed
    if dataset == "ciciot2023":
        return _load_ciciot_multifile(subsample=subsample, seed=seed)
    if dataset == "ton_iot":
        return _load_ton(subsample=subsample, seed=seed)
    raise ValueError(dataset)


def _find_capture_root(start: Path) -> Path:
    """Locate the directory that directly contains the per-attack capture folders.
    Some unzips nest everything under an extra wrapper (e.g. .../CICIoT2023_multifile/data/),
    so descend through single-child wrapper folders until we find folders that
    actually contain .csv files. Robust to either layout."""
    cur = Path(start)
    for _ in range(4):  # bounded descent
        subdirs = [d for d in cur.iterdir() if d.is_dir()]
        # does any subdir directly hold csvs? then `cur` is the capture root.
        if any(any(s.glob("*.csv")) for s in subdirs):
            return cur
        # otherwise, if there's exactly one wrapper subdir, descend into it
        if len(subdirs) == 1:
            cur = subdirs[0]; continue
        break
    return cur


def _load_ciciot_multifile(subsample: bool, seed: int) -> pd.DataFrame:
    root = _find_capture_root(PATHS.data("raw", "CICIoT2023_multifile"))
    folders = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    if not folders:
        raise FileNotFoundError(f"No capture folders under {root} — check the download path.")

    cap = CFG.get("subsample", {}).get("majority_cap_per_class", 200_000)
    rng = np.random.default_rng(seed)
    frames = []
    for folder in folders:
        files = sorted(glob.glob(os.path.join(root, folder, "*.csv")))
        parts = []
        for fi, f in enumerate(files):
            df = pd.read_csv(f)
            # capture-order proxy: preserve part-file index then within-file row order
            df["_part"] = fi
            df["_row"] = np.arange(len(df))
            parts.append(df)
        cls = pd.concat(parts, ignore_index=True)
        cls[LABEL_COL] = _canon_label(folder)
        cls[GROUP_COL] = folder                      # provenance = capture folder
        # subsample MAJORITY classes only; keep rare classes whole
        if subsample and len(cls) > cap:
            cls = cls.iloc[rng.choice(len(cls), cap, replace=False)].reset_index(drop=True)
        frames.append(cls)
    df = pd.concat(frames, ignore_index=True)
    return df


def _load_ton(subsample: bool, seed: int) -> pd.DataFrame:
    f = PATHS.data("raw", "TON_IoT", "train_test_network.csv")
    df = pd.read_csv(f)
    # Use the MULTICLASS 'type' as the canonical label. Drop the binary 'label'
    # FIRST, then rename — otherwise renaming 'type'->'label' collides with the
    # existing binary 'label' and creates two identically named columns.
    if "label" in df.columns:
        df = df.drop(columns=["label"])
    df = df.rename(columns={"type": LABEL_COL})
    df[GROUP_COL] = "ton_single_capture"            # TON network ships as one curated file
    # order helpers for uniformity with the multifile loader (TON = single capture, so
    # one part; the 'temporal' split degenerates to a within-class row-order split here.
    # NOTE: the curated TON file may not preserve true capture order, so for TON a
    # random/stratified split is arguably more appropriate — revisit at cross-dataset stage.)
    df["_part"] = 0
    df["_row"] = np.arange(len(df))
    return df


def clean(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Identity-field removal + dedup. MECHANISM-CRITICAL. Done on FEATURES only;
    label/group/order-helper columns are preserved."""
    keep_meta = [c for c in (LABEL_COL, GROUP_COL, "_part", "_row") if c in df.columns]
    drop = [c for c in IDENTITY_FIELDS.get(dataset, []) if c in df.columns]
    df = df.drop(columns=drop)
    # numeric coercion + drop rows with inf/NaN introduced by CICFlowMeter
    feat = [c for c in df.columns if c not in keep_meta]
    df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feat).reset_index(drop=True)
    # dedup on FEATURES + label (near-duplicate flow rows). Keep order helpers out of the key.
    dedup_key = [c for c in df.columns if c not in ("_part", "_row")]
    df = df.drop_duplicates(subset=dedup_key).reset_index(drop=True)
    return df


def group_count_audit(df: pd.DataFrame, label_col: str = LABEL_COL) -> pd.DataFrame:
    """Per-class instance count and capture-group count, with the survival verdict.
    For THIS dataset 'groups' = capture part-files within the class (≈ how many
    chunks the class was captured in). Drives the 34-vs-survivors reality check and
    flags which classes can take the STRICT capture-grouped robustness split."""
    surv = CFG["split"]["fine_type_survival"]
    g = df.groupby(label_col).agg(
        n_instances=("_row", "size"),
        n_capture_parts=("_part", lambda s: s.nunique()),
    )
    g["survives_min_instances"] = g["n_instances"] >= surv["min_train_instances"]
    g["can_strict_group"] = g["n_capture_parts"] >= surv["min_groups"]   # >=2 parts
    return g.sort_values("n_instances")


def temporal_within_capture_split(df, seed: int, label_col: str = LABEL_COL):
    """PRIMARY split. Within each (class), order by capture (_part,_row) and cut
    chronologically into train/val/test by config fractions. Near-duplicate bursts
    stay on one side; every class appears on all sides. Returns index arrays."""
    tr, va, te = (CFG["split"][k] for k in ("train_frac", "val_frac", "test_frac"))
    train_idx, val_idx, test_idx = [], [], []
    for _, sub in df.groupby(label_col):
        order = sub.sort_values(["_part", "_row"]).index.to_numpy()
        n = len(order); a, b = int(n * tr), int(n * (tr + va))
        train_idx += order[:a].tolist(); val_idx += order[a:b].tolist(); test_idx += order[b:].tolist()
    return {"train": np.array(train_idx), "val": np.array(val_idx), "test": np.array(test_idx)}


def strict_capture_grouped_split(df, seed: int, label_col: str = LABEL_COL):
    """ROBUSTNESS split. Whole capture-parts to one side; only meaningful for classes
    with >= 2 parts (others land entirely in train and are reported as excluded from
    this check). Returns index arrays + the list of classes actually evaluated."""
    surv = CFG["split"]["fine_type_survival"]
    rng = np.random.default_rng(seed)
    train_idx, test_idx, evaluated = [], [], []
    for cls, sub in df.groupby(label_col):
        parts = sorted(sub["_part"].unique())
        if len(parts) >= surv["min_groups"]:
            rng.shuffle(parts)
            k = max(1, int(len(parts) * CFG["split"]["test_frac"]))
            test_parts = set(parts[:k]); evaluated.append(cls)
            test_idx += sub[sub["_part"].isin(test_parts)].index.tolist()
            train_idx += sub[~sub["_part"].isin(test_parts)].index.tolist()
        else:
            train_idx += sub.index.tolist()   # single-capture class: train-only here
    return {"train": np.array(train_idx), "test": np.array(test_idx), "evaluated_classes": evaluated}


def random_reference_split(df, seed: int, label_col: str = LABEL_COL):
    """REFERENCE ONLY — stratified random split to MEASURE the leakage gap vs the
    temporal split. Never the headline regime."""
    from sklearn.model_selection import train_test_split
    tr, va, te = (CFG["split"][k] for k in ("train_frac", "val_frac", "test_frac"))
    idx = np.arange(len(df)); y = df[label_col].to_numpy()
    tr_i, tmp_i = train_test_split(idx, train_size=tr, stratify=y, random_state=seed)
    rel = te / (va + te)
    va_i, te_i = train_test_split(tmp_i, test_size=rel, stratify=y[tmp_i], random_state=seed)
    return {"train": tr_i, "val": va_i, "test": te_i}
'''
open(f"{REPO}/src/data.py","w").write(SRC)
print("wrote", len(SRC), "chars")
print("_find_capture_root present:", "_find_capture_root" in SRC)
print("OK — fixed data.py (auto-detect capture root) written")

wrote 10563 chars
_find_capture_root present: True
OK — fixed data.py (auto-detect capture root) written


In [7]:
import importlib
from src import data
importlib.reload(data)
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
print(data.group_count_audit(df).to_string())

                         n_instances  n_capture_parts  survives_min_instances  can_strict_group
label                                                                                          
Uploading_Attack                1252                1                    True             False
Recon-PingSweep                 2262                1                    True             False
Backdoor_Malware                3215                1                    True             False
XSS                             3846                1                    True             False
SqlInjection                    5244                1                    True             False
CommandInjection                5388                1                    True             False
BrowserHijacking                5788                1                    True             False
DictionaryBruteForce           13062                1                    True             False
DDoS-SlowLoris                 23425    

In [8]:
import importlib
from src import data
importlib.reload(data)
import pandas as pd

# raw (no dedup/clean) vs cleaned, per class
raw = data.load_raw("ciciot2023", subsample=True, seed=0)
raw_counts = raw["label"].value_counts()

clean_df = data.clean(raw.copy(), "ciciot2023")
clean_counts = clean_df["label"].value_counts()

rep = pd.DataFrame({"raw": raw_counts, "deduped": clean_counts}).fillna(0).astype(int)
rep["dropped"] = rep["raw"] - rep["deduped"]
rep["dedup_rate_%"] = (100 * rep["dropped"] / rep["raw"]).round(1)
rep["unique_frac_%"] = (100 * rep["deduped"] / rep["raw"]).round(1)
rep = rep.sort_values("raw")   # rare classes at top — the ones we care about
print(rep.to_string())

print("\n--- summary ---")
print(f"total raw rows:     {rep['raw'].sum():,}")
print(f"total deduped rows: {rep['deduped'].sum():,}")
print(f"overall dedup rate: {100*rep['dropped'].sum()/rep['raw'].sum():.1f}%")
# flag any class that lost most of its rows to duplication
risky = rep[rep["unique_frac_%"] < 50]
print("\nclasses with <50% unique rows (worth a look):")
print(risky.to_string() if len(risky) else "  none — all classes retain >=50% unique flows")

                            raw  deduped  dropped  dedup_rate_%  unique_frac_%
label                                                                         
Uploading_Attack           1252     1252        0           0.0          100.0
Recon-PingSweep            2262     2262        0           0.0          100.0
Backdoor_Malware           3218     3215        3           0.1           99.9
XSS                        3846     3846        0           0.0          100.0
SqlInjection               5245     5244        1           0.0          100.0
CommandInjection           5409     5388       21           0.4           99.6
BrowserHijacking           5859     5788       71           1.2           98.8
DictionaryBruteForce      13064    13062        2           0.0          100.0
DDoS-SlowLoris            23426    23425        1           0.0          100.0
DDoS-HTTP_Flood           28790    28790        0           0.0          100.0
DoS-HTTP_Flood            71861    71485      376   

In [10]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")

# restartwiped the session git identity — set it again
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"

!git add -A
!git commit -q -m "Data layer: real multifile loader (auto-detect capture root), splits, config; 34-way + temporal-within-capture confirmed; group-count + dedup audits pass"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")
print("done")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   0002404..cb97e34  main -> main

done


In [11]:
import os, json, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
PREREG = "logs/anchor_preregistration.json"

p = json.load(open(PREREG))

# --- FREEZE ---
p['frozen_on'] = "2026-06-16"
p['status'] = ("FROZEN 2026-06-16. Data-dependent decisions locked from evidence: 34-way confirmed; "
               "primary split temporal_within_capture, robustness strict_capture_grouped; grouping = capture "
               "provenance; subsample majority_cap=200000 (rare whole); seeds [0,1,2,3,4], anchor 0; dedup "
               "verified non-destructive. Build-dependent nulls (arch params, matched macro-F1, crux threshold) "
               "filled only at their stage. Other changes after this date are DOCUMENTED DEVIATIONS.")
p['frozen_resolved_values'] = {
    "label_granularity": "34_way",
    "split_primary": "temporal_within_capture",
    "split_robustness": "strict_capture_grouped",
    "grouping_variable": "capture_folder/part (per-attack capture provenance)",
    "subsample_majority_cap_per_class": 200000,
    "rare_classes": "kept whole (no subsample)",
    "seeds": [0,1,2,3,4], "anchor_seed": 0,
    "dataset_audit": "all 34 classes survive >=50 instances; 19/34 have >=2 capture parts; overall dedup 17.9%, rare classes ~100% unique",
}
if "label_granularity" in p and isinstance(p["label_granularity"], dict):
    p["label_granularity"]["chosen"] = "34_way"
ep = p.setdefault("evaluation_protocol", {})
ep["primary_split"] = "temporal_within_capture (within each class, cut chronologically by capture order)"
ep["robustness_split"] = "strict_capture_grouped (whole capture parts to one side; classes with >=2 parts only)"
ep["grouping_variable"] = {"chosen": "capture_folder/part (per-attack capture provenance)"}
ep["capture_order_assumption"] = "row order within each capture CSV proxies capture time (stated explicitly)"

json.dump(p, open(PREREG, "w"), indent=2)

# verify it re-parses
q = json.load(open(PREREG))
print("frozen_on:", q["frozen_on"])
print("label_granularity:", q["label_granularity"]["chosen"])
print("split_primary:", q["frozen_resolved_values"]["split_primary"])
print("valid JSON — prereg FROZEN\n")

# --- commit the freeze ---
!git add -A
!git commit -q -m "FREEZE prereg 2026-06-16: 34-way + temporal-within-capture locked from data; build-dependent values deferred to their stages"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")
print("frozen + pushed")

frozen_on: 2026-06-16
label_granularity: 34_way
split_primary: temporal_within_capture
valid JSON — prereg FROZEN

 To https://github.com/anasbiswas1/iot-trust-compression.git
   cb97e34..b19ea10  main -> main

frozen + pushed


In [12]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/train.py — training + evaluation core (thin notebooks call these).

Tempered class weights (sqrt-inverse-frequency) by design: a GENTLE rare-class
boost. Full inverse-frequency or focal loss would artificially prop up the rare
classes and MASK the per-class collapse this paper exists to measure. We want
rare classes honestly fragile at baseline so the collapse-under-compression
effect is real and the diagnostic has something to predict.

Leakage discipline: scaler is fit on TRAIN ONLY. Device auto-detected.
"""
from __future__ import annotations
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score

from .config import CFG, PATHS, set_all_seeds
from . import models as _models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
META = ("label", "group", "_part", "_row")


def feature_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in META]


def tempered_class_weights(y_enc: np.ndarray, n_classes: int, temper: float = 0.5) -> torch.Tensor:
    counts = np.bincount(y_enc, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0
    w = (1.0 / counts) ** temper          # sqrt-inverse-frequency (temper=0.5)
    w = w / w.mean()                       # normalise around 1
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def make_tensors(df, splits, feat_cols, le: LabelEncoder, scaler: StandardScaler):
    out = {}
    for name, idx in splits.items():
        if name not in ("train", "val", "test"):
            continue
        sub = df.loc[idx]
        X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
        y = le.transform(sub["label"].to_numpy())
        out[name] = (torch.tensor(X, dtype=torch.float32),
                     torch.tensor(y, dtype=torch.long))
    return out


def train_model(arch, df, dataset, splits, seed, *, epochs=40, patience=6,
                batch_size=4096, lr=1e-3, compression="M0", arch_kwargs=None,
                save=True, verbose=True):
    """Train one anchor under the given (frozen) split. Saves M0 to Drive.
    Returns (model, info) where info has params, macroF1_val/test, label_encoder, scaler."""
    set_all_seeds(seed)
    feat_cols = feature_columns(df)
    le = LabelEncoder().fit(df["label"].to_numpy())
    n_classes = len(le.classes_)

    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]; Xva, yva = t["val"]

    model = _models.build(arch, len(feat_cols), n_classes, **(arch_kwargs or {})).to(DEVICE)
    w = tempered_class_weights(ytr.numpy(), n_classes)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    best_f1, best_state, since = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        # val macro-F1
        model.eval()
        with torch.no_grad():
            vp = model(Xva.to(DEVICE)).argmax(1).cpu().numpy()
        f1 = f1_score(yva.numpy(), vp, average="macro")
        if verbose:
            print(f"  epoch {ep:02d}  val_macroF1={f1:.4f}")
        if f1 > best_f1 + 1e-4:
            best_f1, best_state, since = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            since += 1
            if since >= patience:
                if verbose: print(f"  early stop @ epoch {ep}")
                break
    model.load_state_dict(best_state)

    # test macro-F1
    Xte, yte = t["test"]
    model.eval()
    with torch.no_grad():
        tp = model(Xte.to(DEVICE)).argmax(1).cpu().numpy()
    test_f1 = f1_score(yte.numpy(), tp, average="macro")

    if save:
        path = PATHS.model(dataset, arch, compression, seed)
        torch.save({"state_dict": model.state_dict(),
                    "classes": list(le.classes_),
                    "feat_cols": feat_cols,
                    "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_},
                   path)
        if verbose: print(f"  saved -> {path}")

    info = {"arch": arch, "params": model.num_params(), "n_classes": n_classes,
            "macroF1_val": float(best_f1), "macroF1_test": float(test_f1),
            "label_encoder": le, "scaler": scaler, "feat_cols": feat_cols, "seed": seed}
    return model, info


@torch.no_grad()
def predict(model, df, splits, le, scaler, feat_cols, which="test"):
    model.eval()
    sub = df.loc[splits[which]]
    X = torch.tensor(scaler.transform(sub[feat_cols].to_numpy(np.float32)), dtype=torch.float32)
    logits = model(X.to(DEVICE)).cpu()
    probs = torch.softmax(logits, 1).numpy()
    y_true = le.transform(sub["label"].to_numpy())
    y_pred = probs.argmax(1)
    return y_true, y_pred, probs


def per_class_recall_table(y_true, y_pred, le) -> pd.DataFrame:
    from .metrics import per_class_recall
    rec = per_class_recall(y_true, y_pred)
    rows = [{"label": le.classes_[c], "recall": rec[c],
             "support": int((y_true == c).sum())} for c in sorted(rec)]
    return pd.DataFrame(rows).sort_values("support")


def seed_null_band(arch, df, dataset, splits, seeds, **kw) -> pd.DataFrame:
    """Train at multiple seeds; per-class recall mean/std across seeds = the NULL band.
    A 'collapse' under compression is only real if it exceeds this band."""
    recs = {}
    for s in seeds:
        m, info = train_model(arch, df, dataset, splits, s, save=(s == seeds[0]),
                              compression="M0", verbose=False, **kw)
        yt, yp, _ = predict(m, df, splits, info["label_encoder"], info["scaler"], info["feat_cols"])
        tab = per_class_recall_table(yt, yp, info["label_encoder"]).set_index("label")["recall"]
        recs[s] = tab
    R = pd.DataFrame(recs)
    R["mean"] = R.mean(1); R["std"] = R.std(1)
    return R
'''
open(f"{REPO}/src/train.py","w").write(SRC)
print("wrote src/train.py:", len(SRC), "chars")
print("OK — train.py written to Drive")

wrote src/train.py: 6202 chars
OK — train.py written to Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO)
sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch
print("cwd:", os.getcwd())
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("split.primary:", CFG['split']['primary'], "| 34-way:", CFG['label_granularity']['chosen'])

Mounted at /content/drive
cwd: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
GPU available: True | Tesla T4
split.primary: temporal_within_capture | 34-way: 34_way


In [5]:
import importlib, sys
# drop any half-loaded versions
for m in list(sys.modules):
    if m.startswith("src"):
        del sys.modules[m]

# import each one alone to see which fails and why
for mod in ["src.config", "src.data", "src.metrics", "src.models", "src.train"]:
    try:
        importlib.import_module(mod)
        print("OK  ", mod)
    except Exception as e:
        import traceback
        print("FAIL", mod, "->", type(e).__name__, e)
        traceback.print_exc()
        break

OK   src.config
FAIL src.data -> OSError [Errno 107] Transport endpoint is not connected


Traceback (most recent call last):
  File "/tmp/ipykernel_3950/1023199212.py", line 10, in <cell line: 0>
    importlib.import_module(mod)
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/data.py", line 28, in <module>
    import pandas as pd
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen impor

In [1]:
# force a clean remount — the mount went stale after the GPU switch
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
print("repo reachable:", os.path.isdir(REPO))
print("data.py reachable:", os.path.exists(f"{REPO}/src/data.py"))
os.chdir(REPO); sys.path.insert(0, REPO)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Mounted at /content/drive
repo reachable: True
data.py reachable: True
GPU: Tesla T4


In [3]:

from src.config import CFG, PATHS, set_all_seeds
print("CFG loaded — split.primary:", CFG['split']['primary'], "| 34-way:", CFG['label_granularity']['chosen'])


CFG loaded — split.primary: temporal_within_capture | 34-way: 34_way


In [5]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/train.py — training + evaluation core (thin notebooks call these).

Tempered class weights (sqrt-inverse-frequency) by design: a GENTLE rare-class
boost. Full inverse-frequency or focal loss would artificially prop up the rare
classes and MASK the per-class collapse this paper exists to measure. We want
rare classes honestly fragile at baseline so the collapse-under-compression
effect is real and the diagnostic has something to predict.

Leakage discipline: scaler is fit on TRAIN ONLY. Device auto-detected.
"""
from __future__ import annotations
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score

from .config import CFG, PATHS, set_all_seeds
from . import models as _models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
META = ("label", "group", "_part", "_row")


def feature_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in META]


def tempered_class_weights(y_enc: np.ndarray, n_classes: int, temper: float = 0.5) -> torch.Tensor:
    counts = np.bincount(y_enc, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0
    w = (1.0 / counts) ** temper          # sqrt-inverse-frequency (temper=0.5)
    w = w / w.mean()                       # normalise around 1
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def make_tensors(df, splits, feat_cols, le: LabelEncoder, scaler: StandardScaler):
    out = {}
    for name, idx in splits.items():
        if name not in ("train", "val", "test"):
            continue
        sub = df.loc[idx]
        X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
        y = le.transform(sub["label"].to_numpy())
        out[name] = (torch.tensor(X, dtype=torch.float32),
                     torch.tensor(y, dtype=torch.long))
    return out


@torch.no_grad()
def _forward_batched(model, X: torch.Tensor, batch_size: int = 8192) -> torch.Tensor:
    """Run a forward pass in batches so a 500k-row split can't OOM the GPU.
    Returns logits on CPU."""
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size].to(DEVICE)
        outs.append(model(xb).cpu())
    return torch.cat(outs, 0)


def train_model(arch, df, dataset, splits, seed, *, epochs=40, patience=6,
                batch_size=4096, lr=1e-3, compression="M0", arch_kwargs=None,
                save=True, verbose=True):
    """Train one anchor under the given (frozen) split. Saves M0 to Drive.
    Returns (model, info) where info has params, macroF1_val/test, label_encoder, scaler."""
    set_all_seeds(seed)
    feat_cols = feature_columns(df)
    le = LabelEncoder().fit(df["label"].to_numpy())
    n_classes = len(le.classes_)

    scaler = StandardScaler().fit(df.loc[splits["train"], feat_cols].to_numpy(np.float32))
    t = make_tensors(df, splits, feat_cols, le, scaler)
    Xtr, ytr = t["train"]; Xva, yva = t["val"]

    model = _models.build(arch, len(feat_cols), n_classes, **(arch_kwargs or {})).to(DEVICE)
    w = tempered_class_weights(ytr.numpy(), n_classes)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    tr_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    best_f1, best_state, since = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        # val macro-F1 (BATCHED forward — never push the whole split at once)
        vp = _forward_batched(model, Xva).argmax(1).numpy()
        f1 = f1_score(yva.numpy(), vp, average="macro")
        if verbose:
            print(f"  epoch {ep:02d}  val_macroF1={f1:.4f}")
        if f1 > best_f1 + 1e-4:
            best_f1, best_state, since = f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            since += 1
            if since >= patience:
                if verbose: print(f"  early stop @ epoch {ep}")
                break
    model.load_state_dict(best_state)

    # test macro-F1 (batched)
    Xte, yte = t["test"]
    tp = _forward_batched(model, Xte).argmax(1).numpy()
    test_f1 = f1_score(yte.numpy(), tp, average="macro")

    if save:
        path = PATHS.model(dataset, arch, compression, seed)
        torch.save({"state_dict": model.state_dict(),
                    "classes": list(le.classes_),
                    "feat_cols": feat_cols,
                    "scaler_mean": scaler.mean_, "scaler_scale": scaler.scale_},
                   path)
        if verbose: print(f"  saved -> {path}")

    info = {"arch": arch, "params": model.num_params(), "n_classes": n_classes,
            "macroF1_val": float(best_f1), "macroF1_test": float(test_f1),
            "label_encoder": le, "scaler": scaler, "feat_cols": feat_cols, "seed": seed}
    return model, info


@torch.no_grad()
def predict(model, df, splits, le, scaler, feat_cols, which="test"):
    sub = df.loc[splits[which]]
    X = torch.tensor(scaler.transform(sub[feat_cols].to_numpy(np.float32)), dtype=torch.float32)
    logits = _forward_batched(model, X)          # BATCHED — safe on 500k rows
    probs = torch.softmax(logits, 1).numpy()
    y_true = le.transform(sub["label"].to_numpy())
    y_pred = probs.argmax(1)
    return y_true, y_pred, probs


def per_class_recall_table(y_true, y_pred, le) -> pd.DataFrame:
    from .metrics import per_class_recall
    rec = per_class_recall(y_true, y_pred)
    rows = [{"label": le.classes_[c], "recall": rec[c],
             "support": int((y_true == c).sum())} for c in sorted(rec)]
    return pd.DataFrame(rows).sort_values("support")


def seed_null_band(arch, df, dataset, splits, seeds, **kw) -> pd.DataFrame:
    """Train at multiple seeds; per-class recall mean/std across seeds = the NULL band.
    A 'collapse' under compression is only real if it exceeds this band."""
    recs = {}
    for s in seeds:
        m, info = train_model(arch, df, dataset, splits, s, save=(s == seeds[0]),
                              compression="M0", verbose=False, **kw)
        yt, yp, _ = predict(m, df, splits, info["label_encoder"], info["scaler"], info["feat_cols"])
        tab = per_class_recall_table(yt, yp, info["label_encoder"]).set_index("label")["recall"]
        recs[s] = tab
    R = pd.DataFrame(recs)
    R["mean"] = R.mean(1); R["std"] = R.std(1)
    return R
'''
open(f"{REPO}/src/train.py","w").write(SRC)
print("wrote src/train.py:", len(SRC), "chars  (batched eval -> no OOM)")
assert "_forward_batched" in SRC
print("OK — fixed train.py written")

wrote src/train.py: 6586 chars  (batched eval -> no OOM)
OK — fixed train.py written


In [7]:
import importlib
from src import data, train, metrics
importlib.reload(data); importlib.reload(train)

model, info = train.train_model(
    "cnn1d", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"channels": (64, 128)})
print(f"\nCNN1D params={info['params']}  val_macroF1={info['macroF1_val']:.4f}  test_macroF1={info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(model, df, splits, info["label_encoder"], info["scaler"], info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, info["label_encoder"])
print("\nPER-CLASS RECALL (M0, CNN1D):")
print(tab.to_string(index=False))

conf = probs.max(1); correct = (yp == yt).astype(int)
print(f"\nECE@15 (overall): {metrics.adaptive_ece(conf, correct, 15):.4f}")
tab.to_csv(PATHS.tables("baseline", "cnn1d_M0_per_class_recall.csv"), index=False)
print("\nsaved per-class recall table to results/tables/baseline/")

  epoch 00  val_macroF1=0.5096
  epoch 01  val_macroF1=0.5228
  epoch 02  val_macroF1=0.5235
  epoch 03  val_macroF1=0.5040
  epoch 04  val_macroF1=0.5501
  epoch 05  val_macroF1=0.5660
  epoch 06  val_macroF1=0.5641
  epoch 07  val_macroF1=0.5220
  epoch 08  val_macroF1=0.5487
  epoch 09  val_macroF1=0.5579
  epoch 10  val_macroF1=0.5700
  epoch 11  val_macroF1=0.5664
  epoch 12  val_macroF1=0.5595
  epoch 13  val_macroF1=0.5689
  epoch 14  val_macroF1=0.5116
  epoch 15  val_macroF1=0.5528
  epoch 16  val_macroF1=0.5574
  early stop @ epoch 16
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__cnn1d__M0__seed0.pt

CNN1D params=29730  val_macroF1=0.5700  test_macroF1=0.5451

PER-CLASS RECALL (M0, CNN1D):
                  label   recall  support
       Uploading_Attack 0.021277      188
        Recon-PingSweep 0.000000      340
       Backdoor_Malware 0.004141      483
                    XSS 0.027730      577
           SqlInjectio

In [8]:
import importlib
from src import train, metrics
importlib.reload(train)

# bigger CNN + more epochs/patience + a touch more LR headroom
model, info = train.train_model(
    "cnn1d", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=80, patience=12, batch_size=4096, lr=1e-3,
    arch_kwargs={"channels": (128, 256)})   # ~4x conv capacity
print(f"\nCNN1D params={info['params']}  val_macroF1={info['macroF1_val']:.4f}  test_macroF1={info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(model, df, splits, info["label_encoder"], info["scaler"], info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, info["label_encoder"])
print("\nPER-CLASS RECALL (M0, bigger CNN1D):")
print(tab.to_string(index=False))
conf = probs.max(1); correct = (yp == yt).astype(int)
print(f"\nECE@15: {metrics.adaptive_ece(conf, correct, 15):.4f}")

  epoch 00  val_macroF1=0.5317
  epoch 01  val_macroF1=0.5203
  epoch 02  val_macroF1=0.5409
  epoch 03  val_macroF1=0.5650
  epoch 04  val_macroF1=0.5590
  epoch 05  val_macroF1=0.5681
  epoch 06  val_macroF1=0.5567
  epoch 07  val_macroF1=0.5604
  epoch 08  val_macroF1=0.5298
  epoch 09  val_macroF1=0.5496
  epoch 10  val_macroF1=0.5517
  epoch 11  val_macroF1=0.5607
  epoch 12  val_macroF1=0.5628
  epoch 13  val_macroF1=0.5750
  epoch 14  val_macroF1=0.5649
  epoch 15  val_macroF1=0.5612
  epoch 16  val_macroF1=0.5736
  epoch 17  val_macroF1=0.5683
  epoch 18  val_macroF1=0.5656
  epoch 19  val_macroF1=0.5635
  epoch 20  val_macroF1=0.5501
  epoch 21  val_macroF1=0.5716
  epoch 22  val_macroF1=0.5670
  epoch 23  val_macroF1=0.5728
  epoch 24  val_macroF1=0.5651
  epoch 25  val_macroF1=0.5662
  early stop @ epoch 25
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__cnn1d__M0__seed0.pt

CNN1D params=108578  val_macroF1=0.5750  tes

In [9]:
import importlib
from src import data, train, metrics
importlib.reload(data); importlib.reload(train)
import pandas as pd, numpy as np

SEEDS = [0, 1]   # null band: two seeds (add a third later if you want a tighter band)

# null band retrains at each seed; seed 0 run (save=True) restores the 29k anchor as saved M0
nb = train.seed_null_band(
    "cnn1d", df, "ciciot2023", splits, seeds=SEEDS,
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"channels": (64, 128)})

# attach support + class tier (floored / measurable / robust) from the mean recall
_, _, _ = None, None, None
support = (df.loc[splits["test"]].groupby("label").size())
nb["support"] = support
def tier(r):
    return "floored" if r < 0.10 else ("robust" if r >= 0.90 else "measurable")
nb["tier"] = nb["mean"].apply(tier)
nb = nb.sort_values("support")

pd.set_option("display.max_rows", 40)
print("PER-CLASS RECALL NULL BAND (CNN1D M0, seeds {}):".format(SEEDS))
print(nb[[*SEEDS, "mean", "std", "support", "tier"]].round(4).to_string())

print("\n--- tier counts ---")
print(nb["tier"].value_counts())
print("\n--- null-band health on the MEASURABLE tier (the headline classes) ---")
meas = nb[nb["tier"] == "measurable"]
print(f"measurable classes: {len(meas)}")
print(f"max seed-to-seed std in measurable tier: {meas['std'].max():.4f}")
print(f"mean std in measurable tier: {meas['std'].mean():.4f}")

nb.to_csv(PATHS.tables("baseline", "cnn1d_M0_null_band.csv"))
print("\nsaved null band -> results/tables/baseline/cnn1d_M0_null_band.csv")

PER-CLASS RECALL NULL BAND (CNN1D M0, seeds [0, 1]):
                              0       1    mean     std  support        tier
label                                                                       
Uploading_Attack         0.0213  0.0745  0.0479  0.0266      188     floored
Recon-PingSweep          0.0000  0.0029  0.0015  0.0015      340     floored
Backdoor_Malware         0.0041  0.0373  0.0207  0.0166      483     floored
XSS                      0.0277  0.0312  0.0295  0.0017      577     floored
SqlInjection             0.0470  0.0686  0.0578  0.0108      787     floored
CommandInjection         0.0630  0.0185  0.0408  0.0222      809     floored
BrowserHijacking         0.0196  0.0161  0.0178  0.0017      869     floored
DictionaryBruteForce     0.3276  0.4434  0.3855  0.0579     1960  measurable
DDoS-SlowLoris           0.9542  0.9559  0.9550  0.0009     3514      robust
DDoS-HTTP_Flood          0.7960  0.8298  0.8129  0.0169     4319  measurable
DoS-HTTP_Flood         

In [10]:
import importlib
from src import train
importlib.reload(train)
import pandas as pd

SEEDS = [0, 1, 2, 3, 4]   # the full pre-registered seed set -> proper null band

nb = train.seed_null_band(
    "cnn1d", df, "ciciot2023", splits, seeds=SEEDS,
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"channels": (64, 128)})

support = df.loc[splits["test"]].groupby("label").size()
nb["support"] = support
def tier(r):
    return "floored" if r < 0.10 else ("robust" if r >= 0.90 else "measurable")
nb["tier"] = nb["mean"].apply(tier)
# the band that matters: how big a drop must exceed to count as real collapse (2*std, ~95%)
nb["null_band_2sigma"] = (2 * nb["std"]).round(4)
nb = nb.sort_values("support")

pd.set_option("display.max_rows", 40)
print("5-SEED NULL BAND (CNN1D M0):")
print(nb[[*SEEDS, "mean", "std", "null_band_2sigma", "support", "tier"]].round(4).to_string())

meas = nb[nb["tier"] == "measurable"]
print(f"\nmeasurable: {len(meas)} | mean std {meas['std'].mean():.4f} | max std {meas['std'].max():.4f}")
print("high-variance measurable classes (2sigma band > 0.15 -- need a bigger drop to count):")
print(meas[meas['null_band_2sigma'] > 0.15][['mean','std','null_band_2sigma']].round(4).to_string())

nb.to_csv(PATHS.tables("baseline", "cnn1d_M0_null_band_5seed.csv"))
print("\nsaved 5-seed null band -> results/tables/baseline/")

5-SEED NULL BAND (CNN1D M0):
                              0       1       2       3       4    mean     std  null_band_2sigma  support        tier
label                                                                                                                 
Uploading_Attack         0.0213  0.0745  0.0000  0.0000  0.0000  0.0191  0.0289            0.0577      188     floored
Recon-PingSweep          0.0000  0.0029  0.0029  0.0029  0.0000  0.0018  0.0014            0.0029      340     floored
Backdoor_Malware         0.0041  0.0373  0.0000  0.0228  0.0062  0.0141  0.0139            0.0279      483     floored
XSS                      0.0277  0.0312  0.0035  0.0139  0.0035  0.0159  0.0117            0.0234      577     floored
SqlInjection             0.0470  0.0686  0.0521  0.0559  0.0394  0.0526  0.0097            0.0195      787     floored
CommandInjection         0.0630  0.0185  0.0915  0.0494  0.0334  0.0512  0.0251            0.0502      809     floored
BrowserHijacking   

In [11]:
import pandas as pd
# reload the saved 5-seed band and finalize the 4-tier stratification
nb = pd.read_csv(PATHS.tables("baseline", "cnn1d_M0_null_band_5seed.csv"), index_col=0)

# refine: split 'measurable' into stable vs unstable-confusable by the 2sigma band
def final_tier(row):
    if row["tier"] == "measurable" and row["null_band_2sigma"] > 0.15:
        return "unstable_confusable"
    return row["tier"]
nb["final_tier"] = nb.apply(final_tier, axis=1)

print("FINAL 4-TIER STRATIFICATION:")
print(nb.groupby("final_tier").size().to_string())
print("\nheadline classes (stable_measurable):")
print(nb[nb["final_tier"]=="measurable"].index.tolist())
print("\nexcluded-from-Δrecall, reported separately:")
print(" unstable_confusable:", nb[nb["final_tier"]=="unstable_confusable"].index.tolist())
print(" floored:", nb[nb["final_tier"]=="floored"].index.tolist())

nb.to_csv(PATHS.tables("baseline", "cnn1d_M0_tiers.csv"))
print("\nsaved final tiers -> results/tables/baseline/cnn1d_M0_tiers.csv")

FINAL 4-TIER STRATIFICATION:
final_tier
floored                 7
measurable             13
robust                 10
unstable_confusable     4

headline classes (stable_measurable):
['DictionaryBruteForce', 'DDoS-HTTP_Flood', 'DoS-HTTP_Flood', 'Recon-PortScan', 'Recon-OSScan', 'DDoS-UDP_Flood', 'Recon-HostDiscovery', 'DoS-SYN_Flood', 'DoS-UDP_Flood', 'DNS_Spoofing', 'MITM-ArpSpoofing', 'VulnerabilityScan', 'BenignTraffic']

excluded-from-Δrecall, reported separately:
 unstable_confusable: ['DDoS-TCP_Flood', 'DDoS-SynonymousIP_Flood', 'DoS-TCP_Flood', 'DDoS-SYN_Flood']
 floored: ['Uploading_Attack', 'Recon-PingSweep', 'Backdoor_Malware', 'XSS', 'SqlInjection', 'CommandInjection', 'BrowserHijacking']

saved final tiers -> results/tables/baseline/cnn1d_M0_tiers.csv


In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Stage 1 baseline: CNN1D M0 anchor (29k, macroF1 0.55), 5-seed null band, 4-tier stratification (13 stable-measurable headline classes); Stage-1 gate passed"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

In [ ]:
from src import data
# df = data.load_raw('ciciot2023'); df = data.clean(df, 'ciciot2023')
# repeat for 'ton_iot'

## Group-count audit -> choose grouping variable

For each candidate (session|capture|device): how many fine types survive with >= min_groups and >= min_train_instances. Pick the grouping that keeps the most fine types evaluable; write it to config.

In [ ]:
# for gv in ['session','capture','device']:
#     print(gv); display(data.group_count_audit(df, gv))
# -> set split.grouping_variable and label_granularity.chosen in config/config.yaml

## Lock + persist the grouped split (+ random reference)

Persist index arrays (locked splits). Keep a random split ONLY as the leakage-gap reference.

In [ ]:
# splits = data.grouped_split(df, CFG['split']['grouping_variable'], CFG['anchor_seed'])
# import numpy as np; np.save(PATHS.arrays('ciciot2023','grouped_test_idx.npy'), splits['test'])

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb00: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)
